### 🧪 CodeCritic Agent Provider Evaluation — Behavior, Latency & Output Metrics

This notebook analyzes agent-level execution logs from the **CodeCritic provider log archive**, focusing on **decision patterns**, **latency characteristics**, and **log generation behavior** across multiple agent roles. It transforms structured output schemas into quantitative metrics that power a suite of targeted visualizations.

#### 🔍 Overview

* **Latency by Agent Type**
  Box plot with jittered points showing runtime variation across generator, discriminator, and stability agents.

* **Log Length KDE Curves**
  Smoothed density plots highlighting the verbosity distribution of logs by agent type.

* **Log Length vs. Latency**
  Scatter plot showing whether verbosity correlates with execution time for different agents.

* **Discriminator Score Distribution**
  Histogram of rounded `score` values from discriminators, revealing confidence skew or threshold artifacts.

* **Decision Frequency**
  Bar chart of how often each decision label (e.g. `accepted`) is assigned during discriminator evaluation.

* **Generator Output vs. Latency**
  Visualization of how response length affects execution time in generative agents.

This notebook provides insight into the **runtime behavior**, **output patterns**, and **evaluation confidence** of the agent provider ecosystem within CodeCritic, and is a valuable tool for debugging, tuning, and validating agent design.


### 🧪 AgentProvider Log Expansion — Full Schema Alignment

This cell parses and expands raw agent log entries into a structured, analysis-ready dataframe by extracting key fields from serialized outputs.

#### 🧠 What this code does:

* Loads a `provider_logs.csv` file and filters to include only rows where `provider_type == "PROVIDER_TYPE.AGENT"`.
* Defines an `AgentType` enum and maps known `provider_id`s to their corresponding agent roles (e.g. generator, discriminator).
* Applies a safe JSON parser to deserialize the `output` column.
* Extracts relevant fields from the parsed `AgentOutputSchema`, including:

  * `decision` — the agent’s classification output
  * `score` — confidence or rating metric
  * `response` — raw output text
  * `log` — structured or extracted rationale
  * `file_path` — reference to source file
* Adds a new `agent_type` column by mapping `provider_id` to its known type.
* Reorders columns to make temporal and logical inspection easier.
* Drops the temporary `output_parsed` helper column after use.

#### 🔍 What you're observing:

* The resulting dataframe, `df_agent_expanded`, now exposes **rich semantic fields** embedded inside the raw log output.
* This structure enables downstream visualizations and statistical summaries across agent types.
* Mapping `provider_id → agent_type` ensures the dataset reflects **semantic role**, not just ID-level identity.
* This is a foundational transformation that enables all subsequent exploratory data analysis (EDA) cells.

Use this cell early to **normalize log data** and unlock field-specific insights across agent execution history.


In [2]:
# 🧪 AgentProvider Log Expansion — Full Schema Alignment

import os, sys, json
import pandas as pd
from pathlib import Path
from enum import Enum

# 📁 Set up paths
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

LOG_PATH = Path("tests/backup/provider_logs.csv")
assert LOG_PATH.exists(), "❌ provider_logs.csv not found"

# 📥 Load log data
df = pd.read_csv(LOG_PATH)

# 🔍 Filter only Agent provider logs
df_agent = df[df["provider_type"] == "PROVIDER_TYPE.AGENT"].copy()

# 🧼 Safe JSON parse for output column
def safe_json_parse(val):
    try:
        return json.loads(val) if isinstance(val, str) else {}
    except Exception:
        return {}

df_agent["output_parsed"] = df_agent["output"].apply(safe_json_parse)

# 🧩 Extract AgentOutputSchema fields
df_agent["decision"] = df_agent["output_parsed"].apply(lambda x: x.get("decision"))
df_agent["score"] = df_agent["output_parsed"].apply(lambda x: x.get("score"))
df_agent["response"] = df_agent["output_parsed"].apply(lambda x: x.get("response"))
df_agent["log"] = df_agent["output_parsed"].apply(lambda x: x.get("log"))
df_agent["file_path"] = df_agent["output_parsed"].apply(lambda x: x.get("file_path"))
df_agent["agent_type"] = df_agent["output_parsed"].apply(lambda x: x.get("agent_type"))

# 🧹 Drop helper column
df_agent_expanded = df_agent.drop(columns=["output_parsed"])

# 📐 Reorder for inspection
column_order = [
    "timestamp", "session_id", "provider_id", "provider_type", "file_log_id", "run_id",
    "called_by_type", "called_by_id", "parent_id", "execution_chain", "file_name", 
    "latency_ms", "output_schema", "config_hash", "input", "output",
    "decision", "score", "log", "response", "file_path", "agent_type"
]
df_agent_expanded = df_agent_expanded[column_order]

# 🧾 Display
pd.set_option("display.max_colwidth", 200)
df_agent_expanded.sort_values("timestamp").head(20)


,timestamp,session_id,provider_id,provider_type,file_log_id,run_id,called_by_type,called_by_id,parent_id,execution_chain,...,output_schema,config_hash,input,output,decision,score,log,response,file_path,agent_type
7,2025-06-16 01:51:50.424303+00:00,1,2,PROVIDER_TYPE.AGENT,1,54b82e85-1a95-4f40-873f-67b1c99093ae,PROVIDER_TYPE.SESSION,1,3aa099c8-5318-4e5f-85dd-b2ddededfd02,"['19e70bc1-c91c-466b-bed9-e67893bb0a32', 'b4ead572-cb94-4c82-b71f-4cedf5288242', '2f2c19bd-b6d0-4b2a-adad-45de50a59881', '3aa099c8-5318-4e5f-85dd-b2ddededfd02', '54b82e85-1a95-4f40-873f-67b1c99093...",...,AgentOutputSchema,4114db3a6b1b3b248a09bb2fcee90284,"{""file_path"": ""C:\\Repos\\codecritic\\working_files\\bad_indentation..__state_2051504163.py"", ""session_id"": 1, ""system"": ""unknown"", ""reason"": ""initialization"", ""steps"": 4, ""retry_count"": 0, ""_last...","{""agent_type"": ""generator"", ""decision"": ""unknown"", ""score"": -1.0, ""response"": ""[CODE]\ndef greet():\n print('Hello')\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n- Corrected indentation for the `print...",unknown,-1.00,- Corrected indentation for the `print` statement to conform with Python's PEP8 guidelines.\n- Ensured the function `greet` is properly defined with the body indented.\n- No tradeoffs were made; t...,[CODE]\ndef greet():\n print('Hello')\n[/CODE]\n\n[CONVERSATION_LOG_ENTRY]\n- Corrected indentation for the `print` statement to conform with Python's PEP8 guidelines.\n- Ensured the function `...,C:\Repos\codecritic\working_files\bad_indentation..__state_2051504163.py,generator
25,2025-06-16 01:51:59.339354+00:00,1,3,PROVIDER_TYPE.AGENT,1,4c2773eb-a3c3-4c35-8dcc-7169cad299b9,PROVIDER_TYPE.SESSION,1,b9e2434f-9adb-4ceb-81c3-b5b78cf54875,"['19e70bc1-c91c-466b-bed9-e67893bb0a32', 'b4ead572-cb94-4c82-b71f-4cedf5288242', '2f2c19bd-b6d0-4b2a-adad-45de50a59881', 'b9e2434f-9adb-4ceb-81c3-b5b78cf54875', '4c2773eb-a3c3-4c35-8dcc-7169cad299...",...,AgentOutputSchema,79ed3415952990718f205b42297cc1f7,"{""file_path"": ""C:\\Repos\\codecritic\\working_files\\temp_state_2051593224__state_2051593325.py"", ""session_id"": 1, ""system"": ""unknown"", ""reason"": ""initialization"", ""steps"": 5, ""retry_count"": 0, ""_...","{""agent_type"": ""discriminator"", ""decision"": ""accepted"", ""score"": 0.95, ""response"": ""[AGENT_DECISION]DECISION_TYPE.ACCEPTED[/AGENT_DECISION]\n[CONVERSATION_LOG_ENTRY]Changes:\nAdded: print('Hell...",accepted,0.95,Changes:\nAdded: print('Hello'),[AGENT_DECISION]DECISION_TYPE.ACCEPTED[/AGENT_DECISION]\n[CONVERSATION_LOG_ENTRY]Changes:\nAdded: print('Hello')[/CONVERSATION_LOG_ENTRY],working_files\temp_agent_2052020744.py,discriminator
36,2025-06-16 01:52:03.234105+00:00,1,4,PROVIDER_TYPE.AGENT,1,ef16237b-14af-4c8e-bff1-17bc4a40b94d,PROVIDER_TYPE.SESSION,1,cf5c85de-b171-4a60-b58d-f9572619881b,"['19e70bc1-c91c-466b-bed9-e67893bb0a32', 'b4ead572-cb94-4c82-b71f-4cedf5288242', '2f2c19bd-b6d0-4b2a-adad-45de50a59881', 'cf5c85de-b171-4a60-b58d-f9572619881b', 'ef16237b-14af-4c8e-bff1-17bc4a40b9...",...,AgentOutputSchema,a45805f143e84f088bfe2f82623ba3cc,"{""file_path"": ""C:\\Repos\\codecritic\\working_files\\temp_state_2052032132__state_2052032281.py"", ""session_id"": 1, ""system"": ""unknown"", ""reason"": ""evaluating"", ""steps"": 6, ""retry_count"": 0, ""_last...","{""agent_type"": ""stability"", ""decision"": ""accepted"", ""score"": 1.0, ""response"": ""[AGENT_DECISION]accept[/AGENT_DECISION]\n[CONVERSATION_LOG_ENTRY]Code passed all stability checks.[/CONVERSATION_LOG_...",accepted,1.00,Code passed all stability checks.,[AGENT_DECISION]accept[/AGENT_DECISION]\n[CONVERSATION_LOG_ENTRY]Code passed all stability checks.[/CONVERSATION_LOG_ENTRY],working_files\temp_state_2052032132__state_2052032281.py,stability
80,2025-06-16 01:52:08.489401+00:00,1,2,PROVIDER_TYPE.AGENT,2,dfc26b28-6d78-4e8a-9e9d-0fdedeaa83ba,PROVIDER_TYPE.SESSION,1,e5f9a9b4-8685-439f-945f-7e7ee8f109ef,"['401a6c89-2c30-426f-86d3-0b14102a4045', '729b948e-9bae-47f5-93be-6cd047a0bf16', 'c007901a-cc90-437b-a5f

### 🚀 Latency by Agent Type

This cell visualizes **execution latency (in milliseconds)** across different agent types using a side-by-side box plot with jittered scatter points for fine-grained insight.

#### 🧠 What this code does:

* Creates a working copy of the dataframe for isolation.
* Normalizes the `agent_type` labels for clean axis labels.
* Iterates over each unique agent type and:

  * Extracts latency values.
  * Adds a `go.Box` trace with:

    * Full boxplot statistics (median, quartiles, whiskers).
    * Jittered individual points overlaid.
    * Color, fill, and marker styling for visibility.
* Uses Plotly’s interactive features to:

  * Hover for exact latency values.
  * Zoom and pan to inspect agent-specific ranges.
* Formats the layout with clear axis titles and dimensions.

#### 🔍 What you're observing:

* Each box summarizes the **latency distribution** of that agent type:

  * The **box** shows the interquartile range (IQR).
  * The **horizontal line** inside represents the median.
  * The **whiskers** extend to non-outlier extrema.
  * The **dots** represent individual invocations (including outliers).
* Use this to:

  * Compare performance across agent roles (e.g. `generator` vs `stability`).
  * Spot outlier spikes (e.g. initialization lag or retry penalties).
  * Identify underperforming agents or latency clusters.

This is a key diagnostic view for **performance tuning** and **latency-aware scheduling** in the agent pipeline.


In [ ]:
import plotly.graph_objects as go

df_latency = df_agent_expanded.copy()

# Normalize agent_type display for cleaner x-axis labels
df_latency["agent_type"] = df_latency["agent_type"].astype(str).str.lower()

fig = go.Figure()

for agent_type in df_latency["agent_type"].dropna().unique():
    subset = df_latency[df_latency["agent_type"] == agent_type]

    fig.add_trace(go.Box(
        y=subset["latency_ms"],
        x=[agent_type] * len(subset),
        name=agent_type,
        boxpoints="all",
        jitter=0.4,
        pointpos=0,  # center on box
        marker=dict(
            color="rgba(0,0,0,0.5)",
            size=6,
            line=dict(width=0.5, color="black")
        ),
        line=dict(width=1),
        fillcolor="rgba(180, 200, 255, 0.5)",
        showlegend=False
    ))

fig.update_layout(
    title="Latency by Agent Type",
    xaxis_title="Agent Type",
    yaxis_title="Latency (ms)",
    template="plotly_white",
    width=1600,
    height=500,
)

fig.show()


### 📐 Log Length Distribution by Agent Type

This cell visualizes the **distribution of log lengths** (measured in characters) for each agent type using kernel density estimation (KDE), giving insight into verbosity patterns across agent behavior.

#### 🧠 What this code does:

* Creates a copy of the dataframe and calculates `log_length` as the character count of each `log` field.
* Iterates over each `agent_type` and:

  * Filters out constant-length logs (zero variance).
  * Uses `scipy.stats.gaussian_kde` to compute a smoothed density curve.
  * Adds a Plotly `go.Scatter` trace with:

    * Shaded fill under the curve.
    * Transparent lines for a soft visual stack.
* Lays out the chart with clear axis labels, a white theme, and a wide format to support long-tailed distributions.

#### 🔍 What you're observing:

* Each curve represents the **distribution shape** of log lengths for a particular agent type.
* Higher peaks = more common log lengths.
* Narrow curves = more consistent logging behavior.
* Flat or wide curves = high variance or occasional verbosity spikes.
* Zero-variance agents (e.g. always emit the same log) are **excluded** for clarity.

This view is especially useful when analyzing **agent verbosity**, **token consumption trends**, and **log audit patterns** across different roles in the system.


In [ ]:
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde

# 🧪 Prepare working copy
df_loglen = df_agent_expanded.copy()
df_loglen["log_length"] = df_loglen["log"].apply(lambda x: len(str(x)) if x else 0)

# 📈 Initialize plot
fig = go.Figure()

# 📊 Add KDE for each agent type (only if variance exists)
for agent_type in df_loglen["agent_type"].dropna().unique():
    subset = df_loglen[df_loglen["agent_type"] == agent_type]
    log_lengths = subset["log_length"]

    if log_lengths.nunique() <= 1:
        continue  # 🚫 Skip zero-variance groups

    kde = gaussian_kde(log_lengths)
    x_range = np.linspace(log_lengths.min(), log_lengths.max(), 200)
    y_vals = kde(x_range)

    fig.add_trace(go.Scatter(
        x=x_range,
        y=y_vals,
        fill="tozeroy",
        name=str(agent_type),
        mode="lines",
        opacity=0.5
    ))

# 🎨 Layout
fig.update_layout(
    title="Log Length KDE by Agent Type",
    xaxis_title="Log Length (chars)",
    yaxis_title="Density",
    template="plotly_white",
    width=1600,
    height=550,
    legend_title="Agent Type"
)

fig.show()


### 📊 Log Length vs Latency by Agent Type

This cell visualizes the **relationship between log length and execution latency** across different agent types using a color-coded scatter plot.

#### 🧠 What this code does:

* Copies the full agent log dataframe and calculates `log_length` as the number of characters in each log entry.
* Uses Plotly Express to create a scatter plot where:

  * The x-axis shows log length (verbosity).
  * The y-axis shows execution latency in milliseconds.
  * Each point represents a single agent invocation.
  * Colors distinguish between different `agent_type`s.
* Enhances visibility with:

  * Uniform marker size and transparency
  * Black borders around points
  * Interactive hover showing `provider_id` and `file_log_id`

#### 🔍 What you're observing:

* Vertical clusters indicate agents with **fixed-length logs but variable latencies**.
* Horizontal clusters suggest **latency-stable agents** that vary in verbosity.
* Diagonal or wide spreads reflect agents where **log length correlates with processing time** — potentially due to:

  * Complex tasks generating detailed logs
  * GPT-based agents returning full conversations
* Overlapping clusters help identify shared behaviors or implementation patterns.

This visualization is helpful for diagnosing **performance bottlenecks**, understanding **log overhead**, and verifying whether logging verbosity scales with task complexity.


In [ ]:
import plotly.express as px

df_stability = df_agent_expanded.copy()
df_stability["log_length"] = df_stability["log"].apply(lambda x: len(str(x)) if x else 0)

fig = px.scatter(
    df_stability,
    x="log_length",
    y="latency_ms",
    color="agent_type",
    title="Log Length vs Latency by Agent Type",
    labels={
        "log_length": "Log Length (chars)",
        "latency_ms": "Latency (ms)",
        "agent_type": "Agent Type"
    },
    hover_data=["provider_id", "file_log_id"],
    template="plotly_white"
)

fig.update_traces(marker=dict(size=8, opacity=0.7, line=dict(width=0.5, color='black')))

fig.update_layout(
    width=1600,
    height=550,
    legend_title="Agent Type"
)

fig.show()


### 📈 Discriminator Agent Score Distribution

This cell visualizes the **distribution of scores** produced by discriminator agents, helping identify trends in model judgments and scoring thresholds.

#### 🧠 What this code does:

* Filters the dataframe to include only entries from agents labeled as `"discriminator"`.
* Rounds each score to 3 decimal places and converts to string for consistent binning.
* Sorts bins numerically to ensure a logical left-to-right score progression.
* Uses Plotly Express to generate a histogram with:

  * Rounded scores on the x-axis
  * Count of occurrences on the y-axis
  * Clear bar outlines and moderate opacity for readability

#### 🔍 What you're observing:

* Each bar represents the number of times a specific rounded score was assigned.
* Clusters near 1.000 may suggest **overconfident scoring** or successful convergence.
* Flat distributions may indicate **diverse evaluation behavior** across prompts or iterations.
* Gaps between score levels may reflect **discrete evaluation thresholds** or model artifacts.
* A spike at or near 0.0 may indicate **automatic rejections** or failed evaluations.

This plot is especially useful when auditing **discriminator selectivity**, **score calibration**, or the **effectiveness of decision-making agents** across an experiment.


In [ ]:
import plotly.express as px
import numpy as np

df_score = df_agent_expanded.copy()
df_score = df_score[df_score["agent_type"] == "discriminator"]

# Round and convert to string for categorical axis
df_score["score_bin"] = df_score["score"].round(3).astype(str)

# Sort score bins for display order
score_order = sorted(df_score["score_bin"].unique(), key=lambda x: float(x))

fig = px.histogram(
    df_score,
    x="score_bin",
    category_orders={"score_bin": score_order},
    title="Discriminator Agent Score Distribution",
    labels={"score_bin": "Score (rounded to 0.001)"},
    color_discrete_sequence=["#2CA02C"],
)

fig.update_traces(marker_line_color='white', marker_line_width=1.5, opacity=0.85)

fig.update_layout(
    bargap=0.15,
    yaxis_title="Count",
    xaxis_title="Score (Rounded)",
    width=1600,
    height=500,
    template="plotly_white",
    showlegend=False
)

fig.show()


### 🗳️ Discriminator Agent Decisions

This cell visualizes the **distribution of decision labels** produced by discriminator agents, providing insight into agent behavior across evaluation cycles.

#### 🧠 What this code does:

* Filters the dataframe to include only discriminator agent runs.
* Plots the frequency of each unique `decision` label using Plotly Express:

  * `x` axis shows the decision categories (e.g. `accepted`, `rejected`)
  * `y` axis shows the total count per decision
* Applies styling with:

  * A soft blue palette
  * Wide layout for clarity
  * No legend (since color encodes only one variable)

#### 🔍 What you're observing:

* A single tall bar indicates **uniform decision behavior** — possibly due to:

  * Overconfident discriminator scoring
  * Unbalanced or overfitted input conditions
* Multiple bars suggest **diverse outcomes**, which can be healthy in:

  * Early iterations of training
  * Edge case evaluations
  * Prompt sensitivity exploration
* Skewed distributions may reveal **biases in generation quality** or **flaws in decision thresholds**.

This chart is a fast diagnostic for evaluating **decision diversity**, **discriminator conservativeness**, or detecting stuck behaviors in the agent lifecycle.


In [ ]:
import plotly.express as px

df_decision = df_agent_expanded.copy()
df_decision = df_decision[df_decision["agent_type"] == "discriminator"]

fig = px.histogram(
    df_decision,
    x="decision",
    color_discrete_sequence=["#AEC6CF"],
    title="Discriminator Agent Decisions",
    labels={"decision": "Decision"},
)

fig.update_layout(
    xaxis_title="Decision",
    yaxis_title="Count",
    template="plotly_white",
    width=1600,
    height=450,
    showlegend=False
)

fig.show()


### ⏱️ Generator Response Length vs. Latency

This cell visualizes the relationship between the **length of responses generated by agents** and the **latency required to produce them**, helping uncover performance scaling trends.

#### 🧠 What this code does:

* Filters the dataset to include only generator agent entries.
* Calculates `response_length` by measuring the character length of each generated response.
* Uses Plotly Express to create a scatter plot with:

  * `response_length` on the x-axis
  * `latency_ms` on the y-axis
  * Hoverable metadata for `provider_id` and `file_log_id`
* Applies custom styling:

  * Semi-transparent blue markers with black outlines
  * Clean numeric formatting for large values
  * A white layout for visual clarity

#### 🔍 What you're observing:

* A **positive correlation** may indicate that longer generations take more time — typical for LLM-backed agents.
* **Flat horizontal clusters** may suggest consistent latency regardless of output size, possibly due to:

  * Cached responses
  * Task-level batching
* Outlier points at high latency + low length may indicate:

  * Backend retries
  * Tokenization overhead
  * Slow prompt formatting

This view is essential for understanding **generation efficiency**, **scaling performance**, and for identifying potential **bottlenecks in output synthesis**.


In [ ]:
import plotly.express as px

df_response = df_agent_expanded.copy()
df_response = df_response[df_response["agent_type"] == "generator"]
df_response["response_length"] = df_response["response"].apply(lambda x: len(str(x)) if x else 0)

fig = px.scatter(
    df_response,
    x="response_length",
    y="latency_ms",
    title="Generator Response Length vs. Latency",
    labels={
        "response_length": "Response Length (chars)",
        "latency_ms": "Latency (ms)"
    },
    hover_data=["provider_id", "file_log_id"],
    template="plotly_white"
)

fig.update_traces(marker=dict(size=8, color="#636EFA", opacity=0.6, line=dict(width=1, color='DarkSlateGrey')))

fig.update_layout(
    width=1600,
    height=500,
    xaxis=dict(tickformat=","),
    yaxis=dict(tickformat=",")
)

fig.show()
